In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_parquet("../data/samples/sample_1m.parquet")

Step 1: Define feature and label

In [ ]:
label = "click"

num_cols = ["cost", "cpo", "time_since_last_click"]
cat_cols = [f"cat{i}" for i in range(1, 10)] + ["campaign"]

features = num_cols + cat_cols

Step 2: train / test split (by time)

In [ ]:
df = df.sort_values("timestamp")

train = df.iloc[:800000]
test = df.iloc[800000:]

Step 3: Simple baseline (Logistic Regression)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=100))
])

model.fit(train[features], train[label])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test[label], pred)
print("AUC:", auc)

Step 4: XGBoost model (improved)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model.fit(train[features], train[label])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test[label], pred)
print("XGB AUC:", auc)

Step 5: Feature Engineering

In [ ]:
# log transform
df["log_cost"] = np.log1p(df["cost"])
df["log_cpo"] = np.log1p(df["cpo"])

# time feature
df["time_since_last_click"] = df["time_since_last_click"].replace(-1, np.nan)
df["has_prev_click"] = df["time_since_last_click"].notnull().astype(int)

In [ ]:
# Re-define features
num_cols = ["log_cost", "log_cpo", "time_since_last_click", "has_prev_click"]
cat_cols = [f"cat{i}" for i in range(1, 10)] + ["campaign"]

features = num_cols + cat_cols

In [ ]:
# Re-split
df = df.sort_values("timestamp")

train = df.iloc[:800000]
test = df.iloc[800000:]

In [ ]:
# campaign CTR encoding
ctr_map = train.groupby("campaign")["click"].mean()

train["campaign_ctr"] = train["campaign"].map(ctr_map)
test["campaign_ctr"] = test["campaign"].map(ctr_map)

features.append("campaign_ctr")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Re-run XGBoost
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model.fit(train[features], train["click"])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test["click"], pred)
print("XGB (FE) AUC:", auc)

Step 6: Parameter Tuning (improve AUC)

In [ ]:
# Re-run XGBoost
model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model.fit(train[features], train["click"])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test["click"], pred)
print("XGB (FE_2) AUC:", auc)

Step 7: Ranking

In [ ]:
# score = CTR × value
test["score"] = pred * test["cpo"]

In [ ]:
test.sort_values("score", ascending=False).head(20)

Step 8: Attribution Modeling

In [ ]:
# Last-touch attribution
df["last_touch"] = np.where(
    (df["conversion"] == 1) & (df["click"] == 1) & (df["click_pos"] == df["click_nb"] - 1),
    1.0,
    0.0
)

In [ ]:
# Linear multi-touch attribution
df["multi_touch_linear"] = np.where(
    (df["conversion"] == 1) & (df["click"] == 1) & (df["click_nb"] > 0),
    1 / df["click_nb"],
    0.0
)

In [ ]:
attr_compare = df.groupby("campaign")[["last_touch", "multi_touch_linear"]].mean()
attr_compare.head(20)

In [ ]:
attr_compare["diff"] = attr_compare["multi_touch_linear"] - attr_compare["last_touch"]
attr_compare.sort_values("diff", ascending=False).head(20)

In [ ]:
attr_compare.sort_values("diff", ascending=True).head(20)

In [ ]:
campaign_budget_view = df.groupby("campaign")[["last_touch", "multi_touch_linear", "cpo", "cost"]].mean()
campaign_budget_view

Step 9: Budget Allocation / ROI

In [ ]:
campaign_budget_view["roi_proxy"] = (
    campaign_budget_view["multi_touch_linear"] / campaign_budget_view["cpo"]
)

campaign_budget_view.sort_values("roi_proxy", ascending=False).head(20)

In [ ]:
top = campaign_budget_view.sort_values("roi_proxy", ascending=False).head(10)
bottom = campaign_budget_view.sort_values("roi_proxy", ascending=True).head(10)

top, bottom

In [ ]:
campaign_budget_view["ctr"] = df.groupby("campaign")["click"].mean()
campaign_budget_view

Step 10: Uplift Modeling

In [ ]:
df["treatment"] = (df["click"] == 1).astype(int)

df = df.sort_values("timestamp")

train = df.iloc[:800000]
test = df.iloc[800000:]

uplift = P(conversion | treated) - P(conversion | control)

In [ ]:
# Rebuild campaign_ctr on the NEW train/test
ctr_map = train.groupby("campaign")["click"].mean()
train["campaign_ctr"] = train["campaign"].map(ctr_map)
test["campaign_ctr"] = test["campaign"].map(ctr_map)

# Rebuild features
num_cols = ["log_cost", "log_cpo", "time_since_last_click", "has_prev_click", "campaign_ctr"]
cat_cols = [f"cat{i}" for i in range(1, 10)] + ["campaign"]
features = num_cols + cat_cols

In [ ]:
# Two training groups
train_t = train[train["treatment"] == 1]
train_c = train[train["treatment"] == 0]

In [ ]:
from xgboost import XGBClassifier

# Train two models
model_t = XGBClassifier(n_estimators=100, max_depth=6)
model_c = XGBClassifier(n_estimators=100, max_depth=6)

model_t.fit(train_t[features], train_t["conversion"])
model_c.fit(train_c[features], train_c["conversion"])

In [ ]:
# Predict uplift
p_t = model_t.predict_proba(test[features])[:, 1]
p_c = model_c.predict_proba(test[features])[:, 1]

uplift = p_t - p_c

In [ ]:
# Use uplift for ranking
test["uplift_score"] = uplift

test.sort_values("uplift_score", ascending=False).head(20)

In [ ]:
# Compare CTR vs Uplift
test["ctr_score"] = pred

test[["ctr_score", "uplift_score"]].corr()

In [ ]:
# Conversion via uplift
test_sorted = test.sort_values("uplift_score", ascending=False)

top_10 = test_sorted.head(int(len(test)*0.1))
bottom_10 = test_sorted.tail(int(len(test)*0.1))

top_10["conversion"].mean(), bottom_10["conversion"].mean()

In [ ]:
# Ranking via uplift
test_sorted = test.sort_values("uplift_score", ascending=False)

# cumulative conversion gain
test_sorted["cum_conv"] = test_sorted["conversion"].cumsum()

test_sorted